# Evaluasi RAG — RAGAS (Google Colab)

Dijalankan di Colab karena pyarrow di env Windows lokal tidak stabil (segfault).
Di sini pyarrow bersih → RAGAS jalan normal.

**Metrik:**
- Faithfulness — alert setia pada dokumen konteks (tak halusinasi)
- Context Precision — dokumen retrieval relevan → **dengan vs tanpa rerank**

**Judge:** OpenRouter GPT-4o-mini (dev) / GPT-4o (final) — berbeda dari generator (Nemotron).

**Langkah:** (1) install → (2) upload `rag_eval_data.json` → (3) masukkan API key → (4) run.

## 1. Install

In [ ]:
# Pin versi persis seperti env lokal yang terbukti jalan.
# Kunci: langchain-community==0.3.31 (versi Colab terbaru menghapus chat_models.vertexai
# yang masih diimpor ragas 0.4.3). + sentence-transformers utk Response Relevancy.
!pip install -q "ragas==0.4.3" "langchain==0.3.30" "langchain-core==0.3.86" "langchain-community==0.3.31" "langchain-openai==0.3.35" "instructor==1.15.4" "sentence-transformers"
# PENTING: setelah sel ini, Runtime -> Restart session, lalu jalankan ulang dari sini.

## 2. Upload `rag_eval_data.json`
File ada di lokal: `DeepL_FP/results/rag_eval_data.json`. Jalankan sel lalu pilih file itu.

In [ ]:
from google.colab import files
import json
up = files.upload()
fname = list(up.keys())[0]
data = json.loads(up[fname].decode('utf-8'))
print(f'{len(data)} sampel dimuat dari {fname}')

## 3. Konfigurasi judge (OpenRouter)
Untuk run final ganti `JUDGE_MODEL` ke `openai/gpt-4o`.

In [ ]:
from getpass import getpass
OPENROUTER_API_KEY = getpass('OpenRouter API key: ')
JUDGE_MODEL = 'openai/gpt-4o-mini'   # ganti ke 'openai/gpt-4o' untuk run final laporan

## 4. Jalankan RAGAS

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from ragas import EvaluationDataset, evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import Faithfulness, ResponseRelevancy, LLMContextPrecisionWithoutReference
from langchain_openai import ChatOpenAI
from langchain_core.embeddings import Embeddings
from sentence_transformers import SentenceTransformer

judge = LangchainLLMWrapper(ChatOpenAI(
    api_key=OPENROUTER_API_KEY, base_url='https://openrouter.ai/api/v1',
    model=JUDGE_MODEL, temperature=0))

# Embedding all-MiniLM-L6-v2 (sama dgn sistem retrieval) via sentence-transformers langsung
_st = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
class STEmb(Embeddings):
    def embed_documents(self, texts): return _st.encode(list(texts)).tolist()
    def embed_query(self, text): return _st.encode(text).tolist()
emb = LangchainEmbeddingsWrapper(STEmb())

def mean(res, *keys):
    for k in keys:
        vals = [s[k] for s in res.scores if k in s and s[k] is not None]
        if vals:
            return sum(vals)/len(vals)
    return None

# Generation: Faithfulness + Response Relevancy
print('[1/3] Faithfulness + Response Relevancy (generation)...')
gen = EvaluationDataset.from_list([
    {'user_input': d['query'], 'retrieved_contexts': d['contexts_with_rerank'], 'response': d['answer']}
    for d in data])
gres = evaluate(gen, metrics=[Faithfulness(), ResponseRelevancy()], llm=judge, embeddings=emb)
faith = mean(gres, 'faithfulness')
relev = mean(gres, 'answer_relevancy', 'response_relevancy')

# Retrieval: Context Precision dengan vs tanpa rerank
def ctx(key):
    return EvaluationDataset.from_list([
        {'user_input': d['query'], 'retrieved_contexts': d[key], 'response': d['answer']} for d in data])
K = 'llm_context_precision_without_reference'
print('[2/3] Context Precision DENGAN rerank ...')
cp_wr = mean(evaluate(ctx('contexts_with_rerank'), metrics=[LLMContextPrecisionWithoutReference()], llm=judge), K)
print('[3/3] Context Precision TANPA rerank ...')
cp_nr = mean(evaluate(ctx('contexts_no_rerank'), metrics=[LLMContextPrecisionWithoutReference()], llm=judge), K)

print('\n' + '='*58)
print('            HASIL EVALUASI RAGAS')
print('='*58)
print(f'Sampel      : {len(data)} window anomali')
print(f'Judge       : {JUDGE_MODEL}')
print('-'*58)
print('GENERATION:')
print(f'  Faithfulness         : {faith:.4f}   (setia pada konteks)')
print(f'  Response Relevancy   : {relev:.4f}   (relevan dgn anomali)')
print('-'*58)
print('RETRIEVAL (Context Precision):')
print(f'  TANPA rerank (cosine): {cp_nr:.4f}')
print(f'  DENGAN rerank        : {cp_wr:.4f}')
d = cp_wr - cp_nr
print(f'  Selisih              : {d:+.4f}  ({"lebih baik" if d>0 else "lebih buruk" if d<0 else "sama"})')
print('='*58)